<a href="https://colab.research.google.com/github/hamzaqarni1/DeepLearning/blob/main/Tutorial_12/Tutorial12_Basic_Autoencoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 1: Visualization & Download Script

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

# --- Set Seeds for Reproducibility ---
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)

# ==============================================================================
# 1. HELPER FUNCTIONS & ARCHITECTURE DEFINITION
# ==============================================================================
def load_and_prep_data(dataset_type='mnist'):
    """Loads and flattens MNIST or Fashion-MNIST to [0, 1] range."""
    if dataset_type == 'fashion':
        (x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()
    else:
        (x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()

    # Normalize to [0, 1] and flatten 28x28 images to 784-dimensional vectors
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    x_train_flat = x_train.reshape((len(x_train), np.prod(x_train.shape[1:])))
    x_test_flat = x_test.reshape((len(x_test), np.prod(x_test.shape[1:])))

    return (x_train_flat, x_test_flat), (x_train, x_test)

def build_autoencoder(input_dim=784, latent_dim=32):
    """Builds a symmetrical deep Autoencoder using the Functional API."""
    inputs = layers.Input(shape=(input_dim,))

    # --- ENCODER ---
    e = layers.Dense(128, activation='relu')(inputs)
    e = layers.Dense(64, activation='relu')(e)
    latent_space = layers.Dense(latent_dim, activation='relu')(e) # Bottleneck

    # --- DECODER ---
    d = layers.Dense(64, activation='relu')(latent_space)
    d = layers.Dense(128, activation='relu')(d)
    outputs = layers.Dense(input_dim, activation='sigmoid')(d) # Sigmoid maps back to [0,1]

    return Model(inputs, outputs, name=f"Autoencoder_Latent_{latent_dim}")

def plot_reconstructions(original, recon, n=8, title="Reconstructions", filename="plot.png"):
    """Plots original vs reconstructed images side-by-side."""
    plt.figure(figsize=(16, 4))
    for i in range(n):
        # Display original
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(original[i].reshape(28, 28), cmap='gray')
        plt.title("Original", fontsize=10, fontweight='bold')
        plt.axis('off')

        # Display reconstruction
        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(recon[i].reshape(28, 28), cmap='gray')
        plt.title("Recon", fontsize=10, fontweight='bold')
        plt.axis('off')

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()
    files.download(filename)

# ==============================================================================
# 2. BASELINE MODEL EXECUTION (MNIST | Latent Dim = 32)
# ==============================================================================
print("\n--- Training Baseline Model (MNIST, Latent Dim = 32) ---")
(x_train_flat, x_test_flat), (_, x_test_orig) = load_and_prep_data('mnist')

base_ae = build_autoencoder(latent_dim=32)
base_ae.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy')

# Train model (Note: x_train_flat is passed as both input and target)
base_history = base_ae.fit(
    x_train_flat, x_train_flat,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test_flat, x_test_flat),
    verbose=0
)

# Predict and visualize baseline
base_recon = base_ae.predict(x_test_flat[:8])
plot_reconstructions(
    x_test_flat[:8], base_recon,
    title="Baseline Autoencoder (MNIST | Latent Dim = 32)",
    filename="base_ae_reconstruction.png"
)

# ==============================================================================
# 3. TASK 1: DIFFERENT COMPRESSION FACTOR (MNIST | Latent Dim = 8)
# ==============================================================================
print("\n--- Executing Task 1: High Compression (MNIST, Latent Dim = 8) ---")
# Compression factor: 784 / 8 = 98x reduction in parameters at the bottleneck
compressed_ae = build_autoencoder(latent_dim=8)
compressed_ae.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy')

comp_history = compressed_ae.fit(
    x_train_flat, x_train_flat,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test_flat, x_test_flat),
    verbose=0
)

comp_recon = compressed_ae.predict(x_test_flat[:8])
plot_reconstructions(
    x_test_flat[:8], comp_recon,
    title="Task 1: High Compression Autoencoder (MNIST | Latent Dim = 8)",
    filename="task1_compression.png"
)

# ==============================================================================
# 4. TASK 2: DIFFERENT DATASET (Fashion-MNIST | Latent Dim = 32)
# ==============================================================================
print("\n--- Executing Task 2: Testing on Fashion-MNIST Dataset ---")
(f_train_flat, f_test_flat), (_, f_test_orig) = load_and_prep_data('fashion')

fashion_ae = build_autoencoder(latent_dim=32)
fashion_ae.compile(optimizer=Adam(learning_rate=1e-3), loss='binary_crossentropy')

fashion_history = fashion_ae.fit(
    f_train_flat, f_train_flat,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(f_test_flat, f_test_flat),
    verbose=0
)

fashion_recon = fashion_ae.predict(f_test_flat[:8])
plot_reconstructions(
    f_test_flat[:8], fashion_recon,
    title="Task 2: Dataset Exploration (Fashion-MNIST | Latent Dim = 32)",
    filename="task2_fashion_mnist.png"
)

print("\nAll models trained successfully! Required plots downloaded.")

TensorFlow Version: 2.20.0

--- Training Baseline Model (MNIST, Latent Dim = 32) ---
11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 576ms/step


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Executing Task 1: High Compression (MNIST, Latent Dim = 8) ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- Executing Task 2: Testing on Fashion-MNIST Dataset ---
29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All models trained successfully! Required plots downloaded.
